# z607 - Features de demanda solicitada (Etapa 8)
Agrega `cust_request_tn` (demanda solicitada, distinta de `tn` vendido) sobre FE604. Un solo cambio: fuente nueva de datos.

In [1]:
!pip install -q polars pyarrow

In [2]:
import os
import polars as pl
import warnings
warnings.filterwarnings("ignore")

In [3]:
PARAM = {
    'experimento': 'FE608',
    'features_path': '/home/ds/datasets/tb_features_FE604.parquet',
    'sellin_zeroes_path': '/home/ds/datasets/sell-in-zeroes.txt'
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/FE608


## 1. Agregar cust_request_tn a nivel product_id x periodo
Mismo criterio que se uso para `tn`: suma sobre todos los clientes.

In [4]:
sellin = pl.read_csv(PARAM['sellin_zeroes_path'], separator=",")

demanda = sellin.group_by(["product_id", "periodo"]).agg(
    pl.col("cust_request_tn").sum().alias("cust_request_tn")
)
print(demanda.height)

31523


## 2. Unir con FE604 y calcular ratio de cumplimiento
`ratio_cumplimiento = tn / cust_request_tn`. Cerca de 1: se vendio lo que se pidio. Menor a 1: hubo demanda no satisfecha (posible quiebre de stock), una senal que `tn` solo no puede dar.

In [5]:
df = pl.read_parquet(PARAM['features_path'])
df = df.join(demanda, on=["product_id", "periodo"], how="left")

df = df.with_columns(
    (pl.col("tn") / (pl.col("cust_request_tn") + 1e-6)).alias("ratio_cumplimiento")
)

## 3. Lag y rolling del ratio (solo hacia atras, mismo criterio que z602)

In [6]:
df = df.sort(["product_id", "periodo"])

df = df.with_columns(
    pl.col("ratio_cumplimiento").shift(1).over("product_id").alias("ratio_cumplimiento_lag1")
)

df = df.with_columns(
    pl.col("ratio_cumplimiento_lag1").rolling_mean(window_size=3, min_periods=1).over("product_id").alias("ratio_cumplimiento_media_3"),
    pl.col("ratio_cumplimiento_lag1").rolling_mean(window_size=12, min_periods=1).over("product_id").alias("ratio_cumplimiento_media_12"),
)

## 4. Guardar

In [7]:
salida = os.path.join(ruta, "tb_features_FE608.parquet")
df.write_parquet(salida)
print(salida)
print(df.shape)

/home/ds/exp/FE608/tb_features_FE608.parquet
(31522, 77)
